In [1]:
import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/dataset/Fake_dev.csv
/kaggle/input/dataset/Fake_train.csv
/kaggle/input/dataset/Fake_test_with_labels.csv
/kaggle/input/dataset/Fake_test_without_labels.csv
/kaggle/input/glove-embeddings/glove.6B.100d.txt


In [2]:
df_train = pd.read_csv('/kaggle/input/dataset/Fake_train.csv')
df_dev = pd.read_csv('/kaggle/input/dataset/Fake_dev.csv')
df_test = pd.read_csv('/kaggle/input/dataset/Fake_test_with_labels.csv')
glove_path = '/kaggle/input/glove-embeddings/glove.6B.100d.txt'

In [3]:
import pandas as pd
import numpy as np
import string

def compute_statistics(df):
    # Combine all the text in the dataframe into one large string
    text = ' '.join(df['text'].astype(str))
    
    # Tokenize the text (split by whitespace and remove punctuation)
    tokens = text.split()
    
    # Remove punctuation and make all tokens lowercase for uniformity
    tokens = [word.strip(string.punctuation).lower() for word in tokens]
    
    # Total words
    total_words = len(tokens)
    
    # Unique words
    unique_words = len(set(tokens))
    
    # Average word length
    avg_word_length = np.mean([len(word) for word in tokens])
    
    # Average sentence length (number of words per sentence)
    sentence_lengths = df['text'].apply(lambda x: len(x.split()))
    avg_sentence_length = np.mean(sentence_lengths)
    
    total_sentences = len(df)
    
    stats = {
        'Total Words': total_words,
        'Unique Words': unique_words,
        'Average Word Length': avg_word_length,
        'Average Sentence Length': avg_sentence_length,
        'Total Sentences': total_sentences
    }
    
    return stats

train_stats = compute_statistics(df_train)
dev_stats = compute_statistics(df_dev)
test_stats = compute_statistics(df_test)

print("Training Set Statistics:", train_stats)
print("Development Set Statistics:", dev_stats)
print("Test Set Statistics:", test_stats)


Training Set Statistics: {'Total Words': 37229, 'Unique Words': 17472, 'Average Word Length': 7.198581750785678, 'Average Sentence Length': 11.430457476205097, 'Total Sentences': 3257}
Development Set Statistics: {'Total Words': 8760, 'Unique Words': 5492, 'Average Word Length': 6.9994292237442925, 'Average Sentence Length': 10.748466257668712, 'Total Sentences': 815}
Test Set Statistics: {'Total Words': 11266, 'Unique Words': 6587, 'Average Word Length': 6.893307296289722, 'Average Sentence Length': 11.055937193326791, 'Total Sentences': 1019}


In [4]:
df_test['label'].value_counts()

label
original    512
Fake        507
Name: count, dtype: int64

In [5]:
df_test.shape

(1019, 2)

In [6]:
df_train['label'].value_counts()

label
original    1658
Fake        1599
Name: count, dtype: int64

In [4]:
import emoji
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
import xgboost as xgb
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [8]:
df_test['label'].value_counts()

label
original    512
Fake        507
Name: count, dtype: int64

In [5]:
malayalam_stopwords = [
    'ആണ്', 'ഇത്', 'എന്ന', 'അത്', 'അവ', 'എങ്ങനെ', 'അവൾ', 'ഞാൻ', 'നിങ്ങൾ',
    'ഇവ', 'പോലും', 'പക്ഷേ', 'എന്ത്', 'അല്ല', 'ഇതിനാൽ', 'വേണം', 'പോലെ', 'ഇനി',
    'അതുകൊണ്ട്', 'നിരവധി', 'കൂടാതെ', 'വിടെ', 'എന്താണ്', 'ഓരോ', 'അത്', 'വാക്കുകൾ'
]

def preprocess(df_train, df_dev, df_test):
    # 1. Remove emojis
    df_train['text'] = df_train['text'].apply(lambda x: emoji.replace_emoji(x, replace=''))
    df_dev['text'] = df_dev['text'].apply(lambda x: emoji.replace_emoji(x, replace=''))
    df_test['text'] = df_test['text'].apply(lambda x: emoji.replace_emoji(x, replace=''))
    
    # 2. Remove HTML tags
    df_train['text'] = df_train['text'].apply(lambda x: re.sub(r'<[^>]*>', '', x))
    df_dev['text'] = df_dev['text'].apply(lambda x: re.sub(r'<[^>]*>', '', x))
    df_test['text'] = df_test['text'].apply(lambda x: re.sub(r'<[^>]*>', '', x))
    
    # 3. Convert text to lowercase
    df_train['text'] = df_train['text'].apply(lambda x: x.lower())
    df_dev['text'] = df_dev['text'].apply(lambda x: x.lower())
    df_test['text'] = df_test['text'].apply(lambda x: x.lower())
    
    # 4. Remove punctuation
    df_train['text'] = df_train['text'].apply(lambda x: ''.join([char for char in x if char not in string.punctuation]))
    df_dev['text'] = df_dev['text'].apply(lambda x: ''.join([char for char in x if char not in string.punctuation]))
    df_test['text'] = df_test['text'].apply(lambda x: ''.join([char for char in x if char not in string.punctuation]))
    
    # 5. Remove stopwords (using custom Malayalam stopwords list)
    df_train['text'] = df_train['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word not in malayalam_stopwords]))
    df_dev['text'] = df_dev['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word not in malayalam_stopwords]))
    df_test['text'] = df_test['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word not in malayalam_stopwords]))
    
    # 6. Remove duplicates
    df_train = df_train.drop_duplicates(subset=['text'])
    df_dev = df_dev.drop_duplicates(subset=['text'])
    #df_test = df_test.drop_duplicates(subset=['text'])
    
    # 7. Manually map labels ('Fake' -> 0, 'Original' -> 1)
    df_train['label'] = df_train['label'].replace({'Fake': 0, 'original': 1})
    df_dev['label'] = df_dev['label'].replace({'Fake': 0, 'original': 1})
    df_test['label'] = df_test['label'].replace({'Fake': 0, 'original': 1})
    
    # Return the processed data
    return df_train, df_dev, df_test

df_train, df_dev, df_test = preprocess(df_train, df_dev, df_test)


<ipython-input-5-9f40e12f1998>:39: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_train['label'] = df_train['label'].replace({'Fake': 0, 'original': 1})
<ipython-input-5-9f40e12f1998>:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['label'] = df_train['label'].replace({'Fake': 0, 'original': 1})
<ipython-input-5-9f40e12f1998>:40: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=Fals

In [10]:
df_train.shape

(3110, 2)

In [11]:
df_dev.shape

(803, 2)

In [12]:
df_test.shape

(1019, 2)

In [13]:
# Function to train models and evaluate them
def run_ml_models(df_train, df_dev, df_test):
    # Vectorizers
    vectorizers = {
        'TF-IDF': TfidfVectorizer(max_features=5000),
        'CountVectorizer': CountVectorizer(max_features=5000)
    }

    # Models
    models = [
        LogisticRegression(max_iter=50000),
        RandomForestClassifier(n_estimators=1000),
        SVC(C=0.8, kernel='linear', gamma=1),
        xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'),
        MultinomialNB(),
        VotingClassifier(estimators=[
            ('lr', LogisticRegression(max_iter=50000)),
            ('rf', RandomForestClassifier(n_estimators=1000)),
            ('svm', SVC(C=0.8, kernel='linear', gamma=1)),
            ('mnb', MultinomialNB())
        ], voting='hard')
    ]

    # evaluate the model and calculate the Macro F1 Score
    def train_model_and_evaluate(model, X_train, y_train, X_test, y_test):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        print(f"Classification Report for {model.__class__.__name__}:")
        print(classification_report(y_test, y_pred))
        print(f"Macro F1 Score for {model.__class__.__name__}: {macro_f1}\n")
        return macro_f1

    # Prepare data
    X_train = df_train['text']
    y_train = df_train['label']
    X_dev = df_dev['text']
    y_dev = df_dev['label']
    X_test = df_test['text']
    y_test = df_test['label']

    # Evaluate each model with each vectorizer
    for vectorizer_name, vectorizer in vectorizers.items():
        print(f"\nUsing {vectorizer_name}...\n")
        
        # Vectorize the data
        X_train_vec = vectorizer.fit_transform(X_train)
        X_dev_vec = vectorizer.transform(X_dev)
        X_test_vec = vectorizer.transform(X_test)
        
        # Train and evaluate models
        for model in models:
            train_model_and_evaluate(model, X_train_vec, y_train, X_test_vec, y_test)

run_ml_models(df_train, df_dev, df_test)



Using TF-IDF...

Classification Report for LogisticRegression:
              precision    recall  f1-score   support

           0       0.77      0.74      0.75       507
           1       0.75      0.78      0.77       512

    accuracy                           0.76      1019
   macro avg       0.76      0.76      0.76      1019
weighted avg       0.76      0.76      0.76      1019

Macro F1 Score for LogisticRegression: 0.7593865322766407

Classification Report for RandomForestClassifier:
              precision    recall  f1-score   support

           0       0.76      0.68      0.72       507
           1       0.71      0.79      0.75       512

    accuracy                           0.74      1019
   macro avg       0.74      0.74      0.74      1019
weighted avg       0.74      0.74      0.74      1019

Macro F1 Score for RandomForestClassifier: 0.7350972833340259

Classification Report for SVC:
              precision    recall  f1-score   support

           0       0.77 

In [6]:
import emoji
import re
import string
from sklearn.metrics import classification_report, f1_score
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import nltk
nltk.download('punkt')

def load_glove_embeddings(glove_path, tokenizer, max_words, embedding_dim=100):
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector

    embedding_matrix = np.zeros((max_words, embedding_dim)) 
    for word, idx in tokenizer.word_index.items():
        if idx >= max_words:
            continue
        embedding_vector = glove_dict.get(word)
        if embedding_vector is not None:
            embedding_matrix[idx] = embedding_vector
    
    return embedding_matrix

# deep learning models (CNN, LSTM, CNN+BiLSTM)
def run_dl_models(df_train, df_dev, df_test, glove_path, max_words=5000, max_sequence_length=200, embedding_dim=100):
    # Prepare data
    X_train = df_train['text']
    y_train = df_train['label']
    X_dev = df_dev['text']
    y_dev = df_dev['label']
    X_test = df_test['text']
    y_test = df_test['label']
    
    # Tokenizer to prepare sequences
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_words)
    tokenizer.fit_on_texts(X_train)
    
    # Prepare sequences for CNN, LSTM, and CNN + BiLSTM (directly using the tokenizer)
    X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_sequence_length)
    X_dev_pad = pad_sequences(tokenizer.texts_to_sequences(X_dev), maxlen=max_sequence_length)
    X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_sequence_length)
    
    # Load GloVe embeddings and prepare the embedding matrix
    embedding_matrix = load_glove_embeddings(glove_path, tokenizer, max_words, embedding_dim)

    # CNN Model
    def cnn_model(input_shape, embedding_matrix=None):
        model = Sequential()
        if embedding_matrix is not None:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1], weights=[embedding_matrix], trainable=False))
        else:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1]))
        model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(GlobalMaxPooling1D())
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # LSTM Model
    def lstm_model(input_shape, embedding_matrix=None):
        model = Sequential()
        if embedding_matrix is not None:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1], weights=[embedding_matrix], trainable=False))
        else:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1]))
        model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # CNN + BiLSTM Model
    def cnn_bilstm_model(input_shape, embedding_matrix=None):
        model = Sequential()
        if embedding_matrix is not None:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1], weights=[embedding_matrix], trainable=False))
        else:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1]))
        model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2)))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    # Evaluate models
    print("Training CNN Model...")
    cnn = cnn_model(X_train_pad.shape, embedding_matrix)
    cnn.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
    y_pred_cnn = cnn.predict(X_test_pad)
    print("CNN Model Classification Report:")
    print(classification_report(y_test, (y_pred_cnn > 0.5)))

    print("Training LSTM Model...")
    lstm = lstm_model(X_train_pad.shape, embedding_matrix)
    lstm.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
    y_pred_lstm = lstm.predict(X_test_pad)
    print("LSTM Model Classification Report:")
    print(classification_report(y_test, (y_pred_lstm > 0.5)))

    print("Training CNN + BiLSTM Model...")
    cnn_bilstm = cnn_bilstm_model(X_train_pad.shape, embedding_matrix)
    cnn_bilstm.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
    y_pred_cnn_bilstm = cnn_bilstm.predict(X_test_pad)
    print("CNN + BiLSTM Model Classification Report:")
    print(classification_report(y_test, (y_pred_cnn_bilstm > 0.5)))

    # Ensemble Model (Averaging the predictions)
    print("Training Ensemble Model...")
    # Average predictions from all models
    ensemble_predictions = (y_pred_cnn + y_pred_lstm + y_pred_cnn_bilstm) / 3
    ensemble_preds_binary = (ensemble_predictions > 0.5).astype(int)
    print("Ensemble Model Classification Report:")
    print(classification_report(y_test, ensemble_preds_binary))

# Run the deep learning models (CNN, LSTM, CNN+BiLSTM, Ensemble) with GloVe
run_dl_models(df_train, df_dev, df_test, glove_path=glove_path)


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Training CNN Model...


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.5350 - loss: 0.6914 - val_accuracy: 0.6239 - val_loss: 0.6264
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6575 - loss: 0.5852 - val_accuracy: 0.6389 - val_loss: 0.6040
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6844 - loss: 0.5624 - val_accuracy: 0.6401 - val_loss: 0.6060
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6913 - loss: 0.5375 - val_accuracy: 0.6451 - val_loss: 0.5977
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6990 - loss: 0.5107 - val_accuracy: 0.6426 - val_loss: 0.5980
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7018 - loss: 0.5014 - val_accuracy: 0.6476 - val_loss: 0.5953
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7069 - loss: 0.4954 - val_accuracy: 0.6426 - val_loss: 0.6018
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7081 - loss: 0.4750 - val_accuracy: 0.6451 - val_loss

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 242ms/step - accuracy: 0.5636 - loss: 0.6738 - val_accuracy: 0.6077 - val_loss: 0.6423
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 231ms/step - accuracy: 0.6091 - loss: 0.6450 - val_accuracy: 0.6127 - val_loss: 0.6252
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 235ms/step - accuracy: 0.6103 - loss: 0.6420 - val_accuracy: 0.6189 - val_loss: 0.6206
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 234ms/step - accuracy: 0.6125 - loss: 0.6128 - val_accuracy: 0.6276 - val_loss: 0.6173
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 234ms/step - accuracy: 0.6320 - loss: 0.6104 - val_accuracy: 0.5554 - val_loss: 0.6225
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 238ms/step - accuracy: 0.6274 - loss: 0.6566 - val_accuracy: 0.6252 - val_loss: 0.6186
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 236ms/step - accuracy: 0.6096 - loss: 0.6170 - val_accuracy: 0.6252 - val_loss: 0.6177
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 241ms/step - accuracy: 0.6320 - loss: 0.6006 - val_accuracy: 0.628

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 15s 223ms/step - accuracy: 0.5581 - loss: 0.6699 - val_accuracy: 0.6276 - val_loss: 0.6480
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 214ms/step - accuracy: 0.6301 - loss: 0.6267 - val_accuracy: 0.6376 - val_loss: 0.6118
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 212ms/step - accuracy: 0.6560 - loss: 0.5787 - val_accuracy: 0.6401 - val_loss: 0.6159
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 209ms/step - accuracy: 0.6884 - loss: 0.5358 - val_accuracy: 0.6376 - val_loss: 0.6249
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 213ms/step - accuracy: 0.6837 - loss: 0.5241 - val_accuracy: 0.6501 - val_loss: 0.6212
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 210ms/step - accuracy: 0.7061 - loss: 0.4938 - val_accuracy: 0.6476 - val_loss: 0.6441
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 215ms/step - accuracy: 0.7027 - loss: 0.4768 - val_accuracy: 0.6476 - val_loss: 0.6395
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 215ms/step - accuracy: 0.7139 - loss: 0.4670 - val_accuracy: 0.652

In [14]:
import emoji
import re
import string
from sklearn.metrics import classification_report, f1_score
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import nltk
nltk.download('punkt')

# Function to load GloVe embeddings and prepare embedding matrix
def load_glove_embeddings(glove_path, tokenizer, max_words, embedding_dim=100):
    # Load GloVe embeddings
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector

    # Create an embedding matrix
    embedding_matrix = np.zeros((max_words, embedding_dim))  # Adjust for your embedding dimension (default is 100)
    for word, idx in tokenizer.word_index.items():
        if idx >= max_words:
            continue
        embedding_vector = glove_dict.get(word)
        if embedding_vector is not None:
            embedding_matrix[idx] = embedding_vector
    
    return embedding_matrix

# Function to create and evaluate deep learning models (CNN, LSTM, CNN+BiLSTM)
def run_dl_models(df_train, df_dev, df_test, glove_path, max_words=5000, max_sequence_length=200, embedding_dim=100):
    # Prepare data
    X_train = df_train['text']
    y_train = df_train['label']
    X_dev = df_dev['text']
    y_dev = df_dev['label']
    X_test = df_test['text']
    y_test = df_test['label']
    
    # Tokenizer to prepare sequences
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_words)
    tokenizer.fit_on_texts(X_train)
    
    # Prepare sequences for CNN, LSTM, and CNN + BiLSTM (directly using the tokenizer)
    X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_sequence_length)
    X_dev_pad = pad_sequences(tokenizer.texts_to_sequences(X_dev), maxlen=max_sequence_length)
    X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_sequence_length)
    
    # Load GloVe embeddings and prepare the embedding matrix
    embedding_matrix = load_glove_embeddings(glove_path, tokenizer, max_words, embedding_dim)

    # CNN Model
    def cnn_model(input_shape, embedding_matrix=None):
        model = Sequential()
        if embedding_matrix is not None:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1], weights=[embedding_matrix], trainable=False))
        else:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1]))
        model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(GlobalMaxPooling1D())
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # LSTM Model
    def lstm_model(input_shape, embedding_matrix=None):
        model = Sequential()
        if embedding_matrix is not None:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1], weights=[embedding_matrix], trainable=False))
        else:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1]))
        model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # CNN + BiLSTM Model
    def cnn_bilstm_model(input_shape, embedding_matrix=None):
        model = Sequential()
        if embedding_matrix is not None:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1], weights=[embedding_matrix], trainable=False))
        else:
            model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_shape[1]))
        model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2)))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model

    # Evaluate models
    print("Training CNN Model...")
    cnn = cnn_model(X_train_pad.shape, embedding_matrix)
    cnn.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
    y_pred_cnn = cnn.predict(X_test_pad)
    print(f"CNN Model F1 Score: {f1_score(y_test, (y_pred_cnn > 0.5), average='macro')}")
    
    print("Training LSTM Model...")
    lstm = lstm_model(X_train_pad.shape, embedding_matrix)
    lstm.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
    y_pred_lstm = lstm.predict(X_test_pad)
    print(f"LSTM Model F1 Score: {f1_score(y_test, (y_pred_lstm > 0.5), average='macro')}")
    
    print("Training CNN + BiLSTM Model...")
    cnn_bilstm = cnn_bilstm_model(X_train_pad.shape, embedding_matrix)
    cnn_bilstm.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
    y_pred_cnn_bilstm = cnn_bilstm.predict(X_test_pad)
    print(f"CNN + BiLSTM Model F1 Score: {f1_score(y_test, (y_pred_cnn_bilstm > 0.5), average='macro')}")

# Run the deep learning models (CNN, LSTM, CNN+BiLSTM) with GloVe
run_dl_models(df_train, df_dev, df_test, glove_path=glove_path)


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Training CNN Model...


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.5663 - loss: 0.6667 - val_accuracy: 0.6264 - val_loss: 0.6285
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6605 - loss: 0.5917 - val_accuracy: 0.6389 - val_loss: 0.6040
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6750 - loss: 0.5624 - val_accuracy: 0.6438 - val_loss: 0.5978
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6883 - loss: 0.5279 - val_accuracy: 0.6488 - val_loss: 0.5968
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7009 - loss: 0.5176 - val_accuracy: 0.6501 - val_loss: 0.5973
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7044 - loss: 0.5028 - val_accuracy: 0.6488 - val_loss: 0.6010
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7184 - loss: 0.4795 - val_accuracy: 0.6476 - val_loss: 0.6056
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7143 - loss: 0.4847 - val_accuracy: 0.6513 - val_loss

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 237ms/step - accuracy: 0.5475 - loss: 0.6721 - val_accuracy: 0.6164 - val_loss: 0.6415
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 227ms/step - accuracy: 0.6163 - loss: 0.6438 - val_accuracy: 0.6214 - val_loss: 0.6326
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 227ms/step - accuracy: 0.6185 - loss: 0.6308 - val_accuracy: 0.6252 - val_loss: 0.6141
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 234ms/step - accuracy: 0.6316 - loss: 0.6181 - val_accuracy: 0.6326 - val_loss: 0.6109
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 227ms/step - accuracy: 0.6328 - loss: 0.6142 - val_accuracy: 0.6326 - val_loss: 0.6132
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 228ms/step - accuracy: 0.6480 - loss: 0.6026 - val_accuracy: 0.6351 - val_loss: 0.6096
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 239ms/step - accuracy: 0.6542 - loss: 0.5894 - val_accuracy: 0.6351 - val_loss: 0.6057
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 236ms/step - accuracy: 0.6705 - loss: 0.5746 - val_accuracy: 0.640

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 15s 221ms/step - accuracy: 0.5790 - loss: 0.6735 - val_accuracy: 0.6264 - val_loss: 0.6422
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 212ms/step - accuracy: 0.5976 - loss: 0.6471 - val_accuracy: 0.6301 - val_loss: 0.6113
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 208ms/step - accuracy: 0.6465 - loss: 0.6018 - val_accuracy: 0.6351 - val_loss: 0.6138
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 208ms/step - accuracy: 0.6736 - loss: 0.5533 - val_accuracy: 0.6488 - val_loss: 0.6047
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 213ms/step - accuracy: 0.6959 - loss: 0.5291 - val_accuracy: 0.6463 - val_loss: 0.6243
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 208ms/step - accuracy: 0.6977 - loss: 0.4999 - val_accuracy: 0.6401 - val_loss: 0.6105
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 207ms/step - accuracy: 0.7106 - loss: 0.4737 - val_accuracy: 0.6376 - val_loss: 0.6487
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 216ms/step - accuracy: 0.7122 - loss: 0.4853 - val_accuracy: 0.648

In [24]:
import emoji
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import classification_report, f1_score
from nltk.tokenize import word_tokenize
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Bidirectional, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import nltk
nltk.download('punkt')
    
# Function to create and evaluate deep learning models (CNN, LSTM, CNN+BiLSTM)
def run_dl_models(df_train, df_dev, df_test, max_words=5000, max_sequence_length=200):
    # Prepare data
    X_train = df_train['text']
    y_train = df_train['label']
    X_dev = df_dev['text']
    y_dev = df_dev['label']
    X_test = df_test['text']
    y_test = df_test['label']
    
    # Vectorization (TF-IDF and CountVectorizer)
    vectorizers = {
        'TF-IDF': TfidfVectorizer(max_features=max_words),
        'CountVectorizer': CountVectorizer(max_features=max_words)
    }

    # Tokenize the text data and pad sequences for LSTM and CNN
    def prepare_sequences(vectorizer, X_train, X_dev, X_test):
        X_train_vec = vectorizer.fit_transform(X_train)
        X_dev_vec = vectorizer.transform(X_dev)
        X_test_vec = vectorizer.transform(X_test)
        
        # Pad sequences for LSTM and CNN
        X_train_pad = pad_sequences(X_train_vec.toarray(), maxlen=max_sequence_length)
        X_dev_pad = pad_sequences(X_dev_vec.toarray(), maxlen=max_sequence_length)
        X_test_pad = pad_sequences(X_test_vec.toarray(), maxlen=max_sequence_length)
        
        return X_train_pad, X_dev_pad, X_test_pad

    # CNN Model
    def cnn_model(input_shape):
        model = Sequential()
        model.add(Embedding(input_dim=max_words, output_dim=128, input_length=input_shape[1]))
        model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(GlobalMaxPooling1D())
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # LSTM Model
    def lstm_model(input_shape):
        model = Sequential()
        model.add(Embedding(input_dim=max_words, output_dim=128, input_length=input_shape[1]))
        model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # CNN + BiLSTM Model
    def cnn_bilstm_model(input_shape):
        model = Sequential()
        model.add(Embedding(input_dim=max_words, output_dim=128, input_length=input_shape[1]))
        model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
        model.add(MaxPooling1D(pool_size=2))
        model.add(Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2)))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        return model
    
    # Evaluate models
    for vectorizer_name, vectorizer in vectorizers.items():
        print(f"\nUsing {vectorizer_name}...\n")
        
        # Prepare sequences for CNN, LSTM, and CNN + BiLSTM
        X_train_pad, X_dev_pad, X_test_pad = prepare_sequences(vectorizer, X_train, X_dev, X_test)
        
        # CNN Model
        print("Training CNN Model...")
        cnn = cnn_model(X_train_pad.shape)
        cnn.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
        y_pred_cnn = cnn.predict(X_test_pad)
        print(f"CNN Model F1 Score: {f1_score(y_test, (y_pred_cnn > 0.5), average='macro')}")
        
        # LSTM Model
        print("Training LSTM Model...")
        lstm = lstm_model(X_train_pad.shape)
        lstm.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
        y_pred_lstm = lstm.predict(X_test_pad)
        print(f"LSTM Model F1 Score: {f1_score(y_test, (y_pred_lstm > 0.5), average='macro')}")
        
        # CNN + BiLSTM Model
        print("Training CNN + BiLSTM Model...")
        cnn_bilstm = cnn_bilstm_model(X_train_pad.shape)
        cnn_bilstm.fit(X_train_pad, y_train, epochs=15, batch_size=64, validation_data=(X_dev_pad, y_dev))
        y_pred_cnn_bilstm = cnn_bilstm.predict(X_test_pad)
        print(f"CNN + BiLSTM Model F1 Score: {f1_score(y_test, (y_pred_cnn_bilstm > 0.5), average='macro')}")

run_dl_models(df_train, df_dev, df_test)


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!

Using TF-IDF...

Training CNN Model...
Epoch 1/15


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.5060 - loss: 0.6939 - val_accuracy: 0.5044 - val_loss: 0.6933
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5024 - loss: 0.6935 - val_accuracy: 0.4956 - val_loss: 0.6932
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4922 - loss: 0.6938 - val_accuracy: 0.5044 - val_loss: 0.6933
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5143 - loss: 0.6928 - val_accuracy: 0.4956 - val_loss: 0.6931
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4929 - loss: 0.6933 - val_accuracy: 0.5068 - val_loss: 0.6934
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4831 - loss: 0.6948 - val_accuracy: 0.5068 - val_loss: 0.6929
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5025 - loss: 0.6931 - val_accuracy: 0.5068 - val_loss: 0.6930
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5105 - loss: 0.6931 - val_accuracy: 0.5068 - val_loss: 0.6929
Ep

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 280ms/step - accuracy: 0.4993 - loss: 0.6944 - val_accuracy: 0.5044 - val_loss: 0.6932
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 273ms/step - accuracy: 0.5055 - loss: 0.6935 - val_accuracy: 0.5044 - val_loss: 0.6932
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 274ms/step - accuracy: 0.4871 - loss: 0.6938 - val_accuracy: 0.5044 - val_loss: 0.6935
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 284ms/step - accuracy: 0.4948 - loss: 0.6938 - val_accuracy: 0.5044 - val_loss: 0.6933
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 282ms/step - accuracy: 0.4831 - loss: 0.6940 - val_accuracy: 0.5044 - val_loss: 0.6937
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 288ms/step - accuracy: 0.5134 - loss: 0.6931 - val_accuracy: 0.4956 - val_loss: 0.6935
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 281ms/step - accuracy: 0.4938 - loss: 0.6933 - val_accuracy: 0.5044 - val_loss: 0.6937
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 286ms/step - accuracy: 0.5109 - loss: 0.6930 - val_accuracy: 0.504

...

49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.4963 - loss: 0.6941 - val_accuracy: 0.5044 - val_loss: 0.6928
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5066 - loss: 0.6932 - val_accuracy: 0.5044 - val_loss: 0.6927
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4979 - loss: 0.6938 - val_accuracy: 0.5044 - val_loss: 0.6943
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5180 - loss: 0.6933 - val_accuracy: 0.5131 - val_loss: 0.6926
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5140 - loss: 0.6931 - val_accuracy: 0.5044 - val_loss: 0.6930
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5031 - loss: 0.6932 - val_accuracy: 0.5131 - val_loss: 0.6925
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5212 - loss: 0.6916 - val_accuracy: 0.5131 - val_loss: 0.6925
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5084 - loss: 0.6920 - val_accuracy: 0.5044 - val_loss: 0.6927
Ep

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 276ms/step - accuracy: 0.4944 - loss: 0.6943 - val_accuracy: 0.5044 - val_loss: 0.6937
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 266ms/step - accuracy: 0.5003 - loss: 0.6943 - val_accuracy: 0.5044 - val_loss: 0.6931
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 266ms/step - accuracy: 0.5162 - loss: 0.6927 - val_accuracy: 0.4981 - val_loss: 0.6932
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 268ms/step - accuracy: 0.4925 - loss: 0.6936 - val_accuracy: 0.5044 - val_loss: 0.6935
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 268ms/step - accuracy: 0.4919 - loss: 0.6937 - val_accuracy: 0.5044 - val_loss: 0.6932
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 270ms/step - accuracy: 0.4929 - loss: 0.6937 - val_accuracy: 0.5044 - val_loss: 0.6931
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 265ms/step - accuracy: 0.4976 - loss: 0.6936 - val_accuracy: 0.5044 - val_loss: 0.6932
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 13s 272ms/step - accuracy: 0.4914 - loss: 0.6937 - val_accuracy: 0.504

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 223ms/step - accuracy: 0.4987 - loss: 0.6942 - val_accuracy: 0.5044 - val_loss: 0.6949
Epoch 2/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 209ms/step - accuracy: 0.4964 - loss: 0.6935 - val_accuracy: 0.4981 - val_loss: 0.6933
Epoch 3/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 211ms/step - accuracy: 0.5039 - loss: 0.6942 - val_accuracy: 0.5056 - val_loss: 0.6925
Epoch 4/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 210ms/step - accuracy: 0.5118 - loss: 0.6923 - val_accuracy: 0.5293 - val_loss: 0.6913
Epoch 5/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 209ms/step - accuracy: 0.5179 - loss: 0.6924 - val_accuracy: 0.5044 - val_loss: 0.6902
Epoch 6/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 214ms/step - accuracy: 0.5034 - loss: 0.6924 - val_accuracy: 0.5118 - val_loss: 0.6897
Epoch 7/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 209ms/step - accuracy: 0.5006 - loss: 0.6904 - val_accuracy: 0.5093 - val_loss: 0.6884
Epoch 8/15
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 209ms/step - accuracy: 0.5099 - loss: 0.6879 - val_accuracy: 0.530